<a href="https://colab.research.google.com/github/gitmystuff/DTSC3010/blob/main/Week_08/Week_08_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A/B Testing Assignment


Your Name

## Getting Started

* Colab - get notebook from the gitmystuff DTSC3010 repository
* Save a Copy in Drive
* Remove Copy of
* Edit your name
* Change the filename to your name - it should be the same as the name in the cell above
* Clean up Colab Notebooks folder
* Submit shared link

### Instructions
1. Answer **all questions** in the provided answer cells. Written answers should be 2–4 sentences.
2. For coding questions, write and run your Python code in the provided cells. Show all outputs.
3. **Be careful with the cell that generates your unique seed** after starting — it will overwrite your unique dataset.
4. Partial credit is awarded for correct methodology even if final numbers differ slightly.

## Setup — Run This Cell First

In [ ]:
import hashlib
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.stats.api as sms
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from math import ceil
import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded successfully.')

## Generate Your Unique Seed

Run the cell below **once** to generate your personal dataset.

In [ ]:
import time

# Your seed is generated from the current time when you run this cell.
# !! IMPORTANT: Make sure the seed is printed below after running. !!

seed = int(time.time() * 1000) % (2**32)
rng  = np.random.default_rng(seed)

print('=' * 50)
print(f'  YOUR SEED: {seed}')
print('=' * 50)
print()
print('Running this cell again will generate a DIFFERENT seed')
print('and change your dataset. Do not re-run after starting.')

In [ ]:
# ── Scenario library ────────────────────────────────────────────────────────
SCENARIOS = [
    dict(
        company='StreamNest', industry='streaming platform',
        test="a redesigned 'Start Free Trial' button with new color and copy",
        metric='free trial sign-up rate',
        unit='website visitor (session)',
        context="StreamNest currently shows a gray 'Start Free Trial' button on their landing page. "
                "The growth team wants to test a vibrant teal button with the copy 'Try Free for 30 Days' "
                "to improve sign-up rates.",
        guardrails=['page load time (must stay under 2s)', 'bounce rate (must not increase by more than 5%)']
    ),
    dict(
        company='CartGo', industry='e-commerce marketplace',
        test='a simplified one-page checkout replacing the current 3-step process',
        metric='checkout completion rate',
        unit='user who initiates checkout',
        context="CartGo's checkout has three separate pages (cart -> shipping -> payment). "
                "The UX team designed a single-page checkout and wants to test if it reduces cart abandonment.",
        guardrails=['average order value (must not decrease)', 'payment error rate (must stay under 1%)']
    ),
    dict(
        company='LearnLoop', industry='edtech SaaS platform',
        test='an in-app guided product tour replacing the current static welcome email',
        metric='7-day activation rate (users completing their first lesson within 7 days)',
        unit='newly registered user',
        context='LearnLoop sends new users a static welcome email. The product team built an in-app '
                'guided tour and wants to test whether it increases the share of users who complete '
                'their first lesson within 7 days of registration.',
        guardrails=['email unsubscribe rate (must not increase)', 'support ticket volume (must not rise significantly)']
    ),
    dict(
        company='PocketBudget', industry='personal finance app',
        test='a prominent dashboard card prompting bank account connection vs. the current Settings-menu prompt',
        metric='bank account connection rate within 48 hours of sign-up',
        unit='newly registered user',
        context='PocketBudget wants more users to link their bank accounts. The current flow hides '
                'the prompt in Settings. The team wants to test a prominent dashboard card prompting '
                'users to connect immediately after sign-up.',
        guardrails=['app crash rate (must not increase)', 'user session length (must not decrease significantly)']
    ),
    dict(
        company='TrekFare', industry='travel booking platform',
        test="a social proof banner showing real-time booking counts on hotel listing pages",
        metric='hotel booking conversion rate',
        unit='user who views a hotel listing page',
        context="TrekFare hypothesizes that showing real-time social proof (e.g., '347 people booked "
                "this hotel today') on hotel listing pages will increase bookings. Treatment users see "
                "the banner; control users do not.",
        guardrails=['average booking value (must not decrease)', 'post-stay customer satisfaction score (must stay stable)']
    ),
    dict(
        company='NourishAI', industry='health and wellness app',
        test='a personalized meal plan shown at sign-up vs. a generic getting-started checklist',
        metric='14-day retention rate (users who log at least 5 meals in 14 days)',
        unit='newly registered user',
        context='NourishAI collects dietary preferences at sign-up. The team wants to test whether '
                'immediately showing a personalized meal plan (vs. a generic checklist) improves '
                'early retention over 14 days.',
        guardrails=['individual meal log completion rate (must not drop)', 'premium upgrade rate (must not be negatively impacted)']
    ),
]

BASE_RATES = [0.08, 0.10, 0.11, 0.13, 0.14, 0.16, 0.18, 0.20, 0.22, 0.25]
LIFTS      = [0.015, 0.02, 0.025, 0.03]
POWERS     = [0.80, 0.85]
ALPHAS     = [0.05, 0.05, 0.05, 0.01]   # weighted toward 0.05

# Draw unique parameters
scenario    = SCENARIOS[rng.integers(len(SCENARIOS))]
base_rate   = float(rng.choice(BASE_RATES))
mde         = float(rng.choice(LIFTS))
power       = float(rng.choice(POWERS))
alpha       = float(rng.choice(ALPHAS))
target_rate = round(base_rate + mde, 4)

# Compute required n (used to simulate results)
es_true      = sms.proportion_effectsize(base_rate, target_rate)
n_per_group  = ceil(sms.NormalIndPower().solve_power(es_true, power=power, alpha=alpha, ratio=1))

# Simulate experiment results
true_lift       = mde * (0.5 + rng.random() * 1.1)
true_ctrl_rate  = float(np.clip(base_rate + rng.uniform(-0.005, 0.005), 0.01, 0.99))
true_treat_rate = float(np.clip(true_ctrl_rate + true_lift, 0.01, 0.99))
x_control       = int(round(true_ctrl_rate  * n_per_group))
x_treatment     = int(round(true_treat_rate * n_per_group))
n_control       = n_per_group
n_treatment     = n_per_group
obs_ctrl_rate   = x_control   / n_control
obs_treat_rate  = x_treatment / n_treatment

print('=' * 65)
print('  YOUR PERSONALIZED A/B TESTING SCENARIO')
print('=' * 65)
print(f"  Company    : {scenario['company']} ({scenario['industry']})")
print(f"  Test       : {scenario['test']}")
print(f"  Metric     : {scenario['metric']}")
print(f"  Rand. Unit : {scenario['unit']}")
print('-' * 65)
print(f'  Baseline conversion rate  : {base_rate*100:.1f}%')
print(f'  Target rate (MDE applied) : {target_rate*100:.1f}%')
print(f'  Minimum Detectable Effect : +{mde*100:.1f}%')
print(f'  Significance level (a)    : {alpha}')
print(f'  Statistical power (1-b)   : {power*100:.0f}%')
print('-' * 65)
print(f"  Context: {scenario['context']}")
print('=' * 65)

---
## Part I — Experiment Design
*Estimated time: ~25 minutes*

---
### Question 1 — Hypotheses

State the **null hypothesis (H0)** and **alternative hypothesis (Ha)** for this experiment using proper statistical notation.

Then state whether this is a **one-tailed or two-tailed test** and justify your choice.

**Your Answer:**

- **H0:** *Write your null hypothesis here (e.g., p_control = p_treatment)*

- **Ha:** *Write your alternative hypothesis here*

- **One-tailed or two-tailed?** *Explain your reasoning*

---
### Question 2 — Randomization Unit & Data Leakage

**(a)** What is the appropriate **randomization unit** for this experiment? What does that mean in practice for this company?

**(b)** Define **data leakage (interference)** and give one specific realistic example of how it could occur in this experiment.

**Your Answer:**

**(a)** *Your randomization unit and justification here*

**(b)** *Definition and specific leakage example for this scenario*

---
### Question 3 — SUTVA Assumptions

State **two specific SUTVA (Stable Unit Treatment Value Assumption)** conditions that must hold for this experiment.

For each:
- State the assumption clearly
- Describe a **realistic scenario that would violate it** in this specific business context

**Your Answer:**

**SUTVA Assumption 1:**
- *Assumption:*
- *Realistic violation:*

**SUTVA Assumption 2:**
- *Assumption:*
- *Realistic violation:*

---
### Question 4 — OEC & Guardrail Metrics

**(a)** What is the **Overall Evaluation Criterion (OEC)** for this experiment? Why is it the right primary metric?

**(b)** Identify **two guardrail metrics** you would monitor. For each:
- Explain what it measures and why it matters
- Specify the threshold that would cause you to **stop the experiment early**

**Your Answer:**

**(a) OEC:** *Your choice and justification*

**(b) Guardrail Metric 1:**
- What it measures and why:
- Stopping threshold:

**(b) Guardrail Metric 2:**
- What it measures and why:
- Stopping threshold:

---
## Part II — Power Analysis & Sample Size
*Estimated time: ~20 minutes*

---
### Question 5 — MDE & Effect Size

**(a)** In your own words, define the **Minimum Detectable Effect (MDE)**. What practical business considerations justify the MDE in your scenario?

**(b)** For proportions, we use **Cohen's h** as our standardized effect size:

$$h = 2 \cdot \arcsin(\sqrt{p_2}) - 2 \cdot \arcsin(\sqrt{p_1})$$

Using `base_rate` (p1) and `target_rate` (p2) from your scenario, compute Cohen's h manually, then verify using `sms.proportion_effectsize()`.

In [ ]:
# Q5(b) -- Cohen's h
p1 = base_rate
p2 = target_rate

# TODO: Calculate Cohen's h manually using the formula above
cohens_h_manual = None

# TODO: Verify using statsmodels
cohens_h_sms = None   # hint: sms.proportion_effectsize(p1, p2)

print(f'Baseline rate  (p1) : {p1:.4f}  ({p1*100:.1f}%)')
print(f'Target rate    (p2) : {p2:.4f}  ({p2*100:.1f}%)')
print(f"Cohen's h (manual)  : {cohens_h_manual}")
print(f"Cohen's h (sms)     : {cohens_h_sms}")

**Q5(a) Written Answer:**

*Your MDE definition and business justification here*

---
### Question 6 — Sample Size Calculation

Using your Cohen's h from Q5, calculate the **required sample size per group** via power analysis.

Use `alpha` and `power` pre-loaded from your scenario.

Then answer:
- What is the **total number of users** needed across both groups?
- What happens to required sample size if you **reduce the MDE**? What about if you **increase power**?

In [ ]:
# Q6 -- Sample size via power analysis

# TODO: Fill in the parameters below
sample_size_per_group = ceil(
    sms.NormalIndPower().solve_power(
        None,        # effect_size -- use cohens_h_sms from Q5
        power=None,  # desired power from your scenario
        alpha=None,  # significance level from your scenario
        ratio=1      # equal group sizes
    )
)

print(f'Alpha (a)              : {alpha}')
print(f'Power (1-b)            : {power}')
print(f"Effect size (Cohen's h): {cohens_h_sms:.6f}")
print(f'Sample size per group  : {sample_size_per_group:,}')
print(f'Total users needed     : {sample_size_per_group * 2:,}')

**Q6 Written Answer:**

*Interpret your result. What happens to required sample size if you reduce the MDE? Increase power? Why?*

---
## Part III — Statistical Analysis
*Estimated time: ~30 minutes*

The experiment has concluded. Run the cell below to see your results.

---

In [ ]:
# Experiment results (unique to your Student ID -- do not modify this cell)
print('=' * 55)
print('  EXPERIMENT RESULTS')
print('=' * 55)
print(f'  Control group   | n = {n_control:>6,} | conversions = {x_control:>5,}')
print(f'  Treatment group | n = {n_treatment:>6,} | conversions = {x_treatment:>5,}')
print('-' * 55)
print(f'  Observed control rate   : {obs_ctrl_rate:.4f}  ({obs_ctrl_rate*100:.2f}%)')
print(f'  Observed treatment rate : {obs_treat_rate:.4f}  ({obs_treat_rate*100:.2f}%)')
print(f'  Observed lift           : {(obs_treat_rate - obs_ctrl_rate)*100:+.2f}%')
print('=' * 55)

---
### Question 7 — Conversion Rates & Pooled Standard Error

**(a)** Calculate the observed conversion rate for each group (verify against the output above).

**(b)** Calculate the **pooled standard error** of the difference between proportions:

$$SE_{pooled} = \sqrt{\hat{p}(1-\hat{p}) \cdot \left(\frac{1}{n_1} + \frac{1}{n_2}\right)}$$

where $\hat{p}$ is the pooled proportion across both groups. Explain what the standard error represents.

In [ ]:
# Q7 -- Conversion rates and pooled standard error

# (a) Observed conversion rates
p_control   = None   # TODO: x_control / n_control
p_treatment = None   # TODO: x_treatment / n_treatment

# (b) Pooled proportion
p_pooled = None      # TODO: (x_control + x_treatment) / (n_control + n_treatment)

# (b) Pooled standard error
se_pooled = None     # TODO: apply the formula above using np.sqrt, e.g. np.sqrt(p_pooled * (1 - p_pooled) * (1/n_control + 1/n_treatment))

print(f'Control conversion rate   : {p_control}')
print(f'Treatment conversion rate : {p_treatment}')
print(f'Pooled proportion (p-hat) : {p_pooled}')
print(f'Pooled standard error     : {se_pooled}')

**Q7 Written Answer:**

*What does the pooled standard error represent in this experiment?*

---
### Question 8 — Hypothesis Test (z-Test of Proportions)

Conduct a **two-sample z-test of proportions**.

The z-statistic is:
$$z = \frac{\hat{p}_2 - \hat{p}_1}{SE_{pooled}}$$

**(a)** Compute the z-statistic manually using your SE from Q7. Then verify using `proportions_ztest`.

**(b)** Report the p-value. Based on your scenario's alpha, do you **reject or fail to reject H0**?

In [ ]:
# Q8 -- z-Test of proportions

# (a) Manual z-statistic
z_stat_manual = None   # TODO: (p_treatment - p_control) / se_pooled

# (a) Verify with statsmodels
successes = [x_control, x_treatment]
n_obs     = [n_control, n_treatment]
z_stat, p_value = proportions_ztest(None, nobs=None)  # TODO: pass successes and n_obs

# (b) Decision
decision = 'Reject H0' if p_value < alpha else 'Fail to reject H0'

print(f'z-statistic (manual)      : {z_stat_manual}')
print(f'z-statistic (statsmodels) : {z_stat:.6f}')
print(f'p-value                   : {p_value:.6f}')
print(f'Alpha (a)                 : {alpha}')
print(f'Decision                  : {decision}')

**Q8 Written Answer:**

*Interpret the p-value in plain language. What does rejecting or failing to reject H0 mean for the business?*

---
### Question 9 — Confidence Intervals & Practical Significance

**(a)** Calculate a **95% confidence interval** for the conversion rate of each group using `proportion_confint`.

**(b)** Do the confidence intervals overlap? What does overlap (or lack thereof) imply about statistical significance?

**(c)** Compare the **lower bound of the treatment CI** to `target_rate` from your scenario. Is the result **practically significant**? What should the business do?

In [ ]:
# Q9 -- Confidence intervals

# 95% CI for each group
ci_control   = proportion_confint(None, None, alpha=0.05, method='normal')  # TODO: x_control,   n_control
ci_treatment = proportion_confint(None, None, alpha=0.05, method='normal')  # TODO: x_treatment, n_treatment

print(f'Control   95% CI : [{ci_control[0]:.4f}, {ci_control[1]:.4f}]  '
      f'({ci_control[0]*100:.2f}% - {ci_control[1]*100:.2f}%)')
print(f'Treatment 95% CI : [{ci_treatment[0]:.4f}, {ci_treatment[1]:.4f}]  '
      f'({ci_treatment[0]*100:.2f}% - {ci_treatment[1]*100:.2f}%)')
print()
print(f'Target rate from scenario : {target_rate:.4f} ({target_rate*100:.1f}%)')
print(f'Treatment CI lower bound  : {ci_treatment[0]:.4f} ({ci_treatment[0]*100:.2f}%)')
practically_sig = ci_treatment[0] >= target_rate
print(f'Practical significance    : CI lower bound >= target rate? {practically_sig}')

**Q9 Written Answer:**

**(b)** *Do the CIs overlap? What does this imply?*

**(c)** *Is the result practically significant? What should the business do?*

---
## Part IV — Interpretation & Conclusions
*Estimated time: ~15 minutes*

---
### Question 10 — Statistical vs. Practical Significance

Explain the difference between **statistical significance** and **practical significance**.

Using your results:
- Is your result statistically significant? How do you know?
- Is your result practically significant? How do you know?
- Can a result be statistically significant but **not** practically significant? Give a brief example.
- Can a result be practically significant but **not** statistically significant? What would that mean for the business?

**Your Answer:**

*Write your full answer here — aim for 6–10 sentences covering all four bullet points*

---
### Question 11 — Final Business Recommendation

Write a short **business memo** (5–8 sentences) to a non-technical stakeholder recommending what to do next.

Your memo should:
- State what was tested and why
- Summarize the key result **with numbers** but without statistical jargon
- Make a clear recommendation: **ship it, don't ship it, or run a follow-up test**
- Note any caveats or risks the stakeholder should be aware of

**Business Memo:**

**To:** Product Team

**Re:** A/B Test Results

*Write your memo here*

## Submission Checklist

Before submitting, confirm:

- [ ] All code cells have been run and show output
- [ ] All `None` values have been replaced with my code
- [ ] All written answer cells have been filled in
- [ ] I verified my reject/fail-to-reject decision against my alpha
- [ ] My business memo (Q11) is written for a non-technical audience
